## Create the operational write-back tables

`fincrime_ops` is a Fabric SQL database, which unlike the mirrored database is
**read-write**. It holds the state the console produces: alert dispositions,
dispute cases and their transitions, KYC actions, and who owns what.

The schema follows `APP_DESIGN.md` section 9. This notebook is generated by
`scripts/build_ops_notebook.py` from the DDL in `scripts/ops_db.py`, so editing
it by hand will be overwritten. Change `ops_db.py` and regenerate.

Every statement guards on existence, so the notebook is safe to re-run.

In [ ]:
import struct
import pyodbc
import notebookutils

SERVER   = "6xdtv76wvqse5lz63juf6hq7gy-bvxa67rqg6pujc7gfpvoykcypm.database.fabric.microsoft.com,1433"
DATABASE = "fincrime_ops-446f966e-7df5-4284-a279-fdb32ef3b8a1"

# The capacity can issue a token for the SQL audience. A local machine running
# the Fabric CLI cannot, because that CLI's cache holds no refresh token.
raw = notebookutils.credentials.getToken("https://database.windows.net/")
tok = raw.encode("utf-16-le")
packed = struct.pack("<i", len(tok)) + tok

con = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};Server=" + SERVER +
    ";Database=" + DATABASE + ";Encrypt=yes;TrustServerCertificate=no",
    attrs_before={1256: packed})   # 1256 = SQL_COPT_SS_ACCESS_TOKEN
con.autocommit = True
cur = con.cursor()
print("connected to", DATABASE)

### The statements

Two tables carry `_mirror_row_id`. No subset of the 100 business columns is
unique, so the surrogate key added for mirroring is the only stable handle back
to a specific source row. It earns its keep a second time here.

`ops.case_event` is append-only on purpose: regulated casework needs the history
of how a case moved, not just where it ended up, and that history is what makes
the SLA clock auditable.

In [ ]:
DDL = [
    "IF SCHEMA_ID('ops') IS NULL EXEC('CREATE SCHEMA ops')",
    "IF OBJECT_ID('ops.alert_disposition') IS NULL\n       CREATE TABLE ops.alert_disposition (\n         disposition_id  BIGINT IDENTITY(1,1) PRIMARY KEY,\n         alert_id        VARCHAR(64)   NOT NULL,\n         _mirror_row_id  BIGINT        NOT NULL,\n         disposition     VARCHAR(32)   NOT NULL,\n         reason          NVARCHAR(512) NULL,\n         analyst         NVARCHAR(256) NOT NULL,\n         decided_at      DATETIME2(3)  NOT NULL CONSTRAINT DF_disp_at DEFAULT SYSUTCDATETIME()\n       )",
    "IF OBJECT_ID('ops.[case]') IS NULL\n       CREATE TABLE ops.[case] (\n         case_id          VARCHAR(64)   NOT NULL PRIMARY KEY,\n         _mirror_row_id   BIGINT        NOT NULL,\n         state            VARCHAR(32)   NOT NULL,\n         owner            NVARCHAR(256) NULL,\n         sla_due_at       DATETIME2(3)  NULL,\n         resolution       VARCHAR(64)   NULL,\n         recovered_amount DECIMAL(18,2) NULL,\n         opened_at        DATETIME2(3)  NOT NULL CONSTRAINT DF_case_at DEFAULT SYSUTCDATETIME()\n       )",
    "IF OBJECT_ID('ops.case_event') IS NULL\n       CREATE TABLE ops.case_event (\n         event_id   BIGINT IDENTITY(1,1) PRIMARY KEY,\n         case_id    VARCHAR(64)   NOT NULL,\n         from_state VARCHAR(32)   NULL,\n         to_state   VARCHAR(32)   NOT NULL,\n         actor      NVARCHAR(256) NOT NULL,\n         note       NVARCHAR(512) NULL,\n         at         DATETIME2(3)  NOT NULL CONSTRAINT DF_evt_at DEFAULT SYSUTCDATETIME()\n       )",
    "IF OBJECT_ID('ops.kyc_action') IS NULL\n       CREATE TABLE ops.kyc_action (\n         action_id       BIGINT IDENTITY(1,1) PRIMARY KEY,\n         customer_id     VARCHAR(64)   NOT NULL,\n         action          VARCHAR(64)   NOT NULL,\n         owner           NVARCHAR(256) NULL,\n         next_review_due DATE          NULL,\n         at              DATETIME2(3)  NOT NULL CONSTRAINT DF_kyc_at DEFAULT SYSUTCDATETIME()\n       )",
    "IF OBJECT_ID('ops.assignment') IS NULL\n       CREATE TABLE ops.assignment (\n         item_type   VARCHAR(32)   NOT NULL,\n         item_id     VARCHAR(64)   NOT NULL,\n         assignee    NVARCHAR(256) NOT NULL,\n         assigned_at DATETIME2(3)  NOT NULL CONSTRAINT DF_asg_at DEFAULT SYSUTCDATETIME(),\n         CONSTRAINT PK_assignment PRIMARY KEY (item_type, item_id)\n       )",
    "IF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name='IX_disp_alert') CREATE INDEX IX_disp_alert ON ops.alert_disposition (alert_id)",
    "IF NOT EXISTS (SELECT 1 FROM sys.indexes WHERE name='IX_evt_case') CREATE INDEX IX_evt_case ON ops.case_event (case_id, at)"
]

for stmt in DDL:
    cur.execute(stmt)
    print("ok:", " ".join(stmt.split())[:78])

print()
print("DDL complete.")

### Verify

Counts rather than a bare success message, so a re-run shows the tables are
present and reports what is in them.

In [ ]:
TABLES = ["ops.alert_disposition", "ops.[case]", "ops.case_event", "ops.kyc_action", "ops.assignment"]

print("%-28s %s" % ("TABLE", "ROWS"))
for t in TABLES:
    cur.execute("SELECT count(*) FROM " + t)
    print("%-28s %d" % (t, cur.fetchone()[0]))

cur.execute("""
    SELECT s.name + '.' + t.name AS tbl, c.name AS col, ty.name AS typ
    FROM sys.tables t
    JOIN sys.schemas s ON s.schema_id = t.schema_id
    JOIN sys.columns c ON c.object_id = t.object_id
    JOIN sys.types ty ON ty.user_type_id = c.user_type_id
    WHERE s.name = 'ops'
    ORDER BY tbl, c.column_id
""")
print()
print("%-26s %-18s %s" % ("TABLE", "COLUMN", "TYPE"))
for r in cur.fetchall():
    print("%-26s %-18s %s" % (r.tbl, r.col, r.typ))

con.close()